# Strategy analysis example

Debugging a strategy can be time-consuming. Freqtrade offers helper functions to visualize raw data.
The following assumes you work with SampleStrategy, data for 5m timeframe from Binance and have downloaded them into the data directory in the default location.
Please follow the [documentation](https://www.freqtrade.io/en/stable/data-download/) for more details.

In [2]:
print('Strategy analysis complete.')


Strategy analysis complete.


## Setup

### Change Working directory to repository root

In [3]:
import os
from pathlib import Path


# Change directory
# Modify this cell to insure that the output shows the correct path.
# Define all paths relative to the project root shown in the cell output
project_root = r"C:\ft_userdata"
i = 0
try:
    os.chdir(project_root)
    if not Path("LICENSE").is_file():
        i = 0
        while i < 4 and (not Path("LICENSE").is_file()):
            os.chdir(Path(Path.cwd(), "../"))
            i += 1
        project_root = Path.cwd()
except FileNotFoundError:
    print("Please define the project root relative to the current directory")
print(Path.cwd())


C:\


In [4]:
import freqtrade

print(freqtrade.__version__)


2026.10


In [5]:
import subprocess

subprocess.run(
    [
        "freqtrade",
        "download-data",
        "--datadir",
        "ft_userdata/user_data/data",
        "--pairs",
        "BTC/USDT",
        "--timeframes",
        "5m",
    ]
)


CompletedProcess(args=['freqtrade', 'download-data', '--datadir', 'ft_userdata/user_data/data', '--pairs', 'BTC/USDT', '--timeframes', '5m'], returncode=1)

In [11]:
import os
from pathlib import Path

# 1. Forcer le répertoire de travail à la racine de votre projet
project_root = r"C:\ft"
os.chdir(project_root)

# Vérification : on doit voir le dossier "user_data" à cet endroit
print("Répertoire de travail :", Path.cwd())
print("user_data existe ?", (Path.cwd() / "user_data").is_dir(), '→', Path.cwd() / "user_data")


Répertoire de travail : C:\ft
user_data existe ? True → C:\ft\user_data


In [ ]:
# from freqtrade.configuration import Configuration
# from pathlib import Path

# # 1. (Optionnel) Chargez votre config principale si vous en avez une
# # config = Configuration.from_files([r"C:\ft_userdata\user_data\config.json", r"C:\ft_userdata\notebook_config.json"])

# # 2. Ou, si vous n'avez pas d'autre fichier, chargez simplement le vôtre
# config = Configuration.from_files([r"C:/ft_userdata/notebook_config.json"])

# # Vérifiez que le chemin est correct
# print(f"user_data_dir utilisé : {config['user_data_dir']}")


### Configure Freqtrade environment

In [13]:
from freqtrade.configuration import Configuration
from pathlib import Path

# Définir le chemin absolu vers votre dossier user_data
# Adaptez ce chemin si nécessaire (par exemple : C:/ft_userdata/user_data)
# user_data_path = Path(r"C:\ft_userdata\user_data").resolve()

# Initialiser la configuration avec le user_data_dir spécifié
config = Configuration.from_files([])
# config["user_data_dir"] = (
#     r"C:\ft\user_data"  # L'ajout de cette ligne résout l'erreur
# )

# Le reste de votre configuration reste inchangé
# config["datadir"] = str(user_data / "data")  # Chemin absolu pour être sûr
config["timeframe"] = "1d"
config["strategy"] = "SampleStrategy"
data_location = config["datadir"]
pair = "BTC/USDT"

print(f"Configuration initialisée. Dossier user_data : {config['user_data_dir']}")


2026-11-05 06:05:19,151 - freqtrade.loggers - INFO - Enabling colorized output.

2026-11-05 06:05:19,154 - freqtrade.loggers - INFO - Logfile configured

2026-11-05 06:05:19,157 - freqtrade.loggers - INFO - Verbosity set to 0

2026-11-05 06:05:19,159 - freqtrade.configuration.configuration - INFO - Using user-data directory: C:\ft\user_data ...

2026-11-05 06:05:19,161 - freqtrade.configuration.configuration - INFO - Using data directory: C:\ft\user_data\data ...

Configuration initialisée. Dossier user_data : C:\ft\user_data


In [14]:
# Load data using values set above
from freqtrade.data.history import load_pair_history
from freqtrade.enums import CandleType


candles = load_pair_history(
    datadir=data_location,
    timeframe=config["timeframe"],
    pair=pair,
    data_format="json",  # Make sure to update this to your data
    candle_type=CandleType.SPOT,
)

# Confirm success
print(f"Loaded {len(candles)} rows of data for {pair} from {data_location}")
candles.head()


2026-11-05 06:05:28,159 - freqtrade.data.history.datahandlers.idatahandler - WARNING - No history for BTC/USDT, spot, 1d found. Use `freqtrade download-data` to download the data

Loaded 0 rows of data for BTC/USDT from C:\ft\user_data\data


,date,open,high,low,close,volume


## Load and run strategy
* Rerun each time the strategy file is changed

In [15]:
# Load strategy using values set above
from freqtrade.data.dataprovider import DataProvider
from freqtrade.resolvers import StrategyResolver


strategy = StrategyResolver.load_strategy(config)
strategy.dp = DataProvider(config, None, None)
strategy.ft_bot_start()

# Generate buy/sell signals using strategy
df = strategy.analyze_ticker(candles, {"pair": pair})
df.tail()


2026-11-05 06:05:41,087 - freqtrade.resolvers.iresolver - INFO - Using resolved strategy SampleStrategy from 'C:\ft\user_data\strategies\sample_strategy.py'...

2026-11-05 06:05:41,089 - freqtrade.strategy.hyper - INFO - Found no parameter file.

2026-11-05 06:05:41,091 - freqtrade.resolvers.strategy_resolver - INFO - Override strategy 'timeframe' with value from the configuration: 1d.

2026-11-05 06:05:41,092 - freqtrade.resolvers.strategy_resolver - INFO - Override strategy 'stake_currency' with value from the configuration: .

2026-11-05 06:05:41,094 - freqtrade.resolvers.strategy_resolver - INFO - Strategy using minimal_roi: {'60': 0.01, '30': 0.02, '0': 0.04}

2026-11-05 06:05:41,095 - freqtrade.resolvers.strategy_resolver - INFO - Strategy using timeframe: 1d

... (23 outputs left out)

2026-11-05 06:05:41,136 - freqtrade.strategy.hyper - INFO - No params for sell found, using default values.

2026-11-05 06:05:41,138 - freqtrade.strategy.hyper - INFO - Strategy Parameter(default): sell_rsi = 70

2026-11-05 06:05:41,140 - freqtrade.strategy.hyper - INFO - Strategy Parameter(default): short_rsi = 70

ValueError: cannot set a frame with no defined index and a scalar

### Display the trade details

* Note that using `data.head()` would also work, however most indicators have some "startup" data at the top of the dataframe.
* Some possible problems
    * Columns with NaN values at the end of the dataframe
    * Columns used in `crossed*()` functions with completely different units
* Comparison with full backtest
    * having 200 buy signals as output for one pair from `analyze_ticker()` does not necessarily mean that 200 trades will be made during backtesting.
    * Assuming you use only one condition such as, `df['rsi'] < 30` as buy condition, this will generate multiple "buy" signals for each pair in sequence (until rsi returns > 29). The bot will only buy on the first of these signals (and also only if a trade-slot ("max_open_trades") is still available), or on one of the middle signals, as soon as a "slot" becomes available.  


In [ ]:
# Report results
print(f"Generated {df['enter_long'].sum()} entry signals")
data = df.set_index("date", drop=False)
data.tail()


## Load existing objects into a Jupyter notebook

The following cells assume that you have already generated data using the cli.  
They will allow you to drill deeper into your results, and perform analysis which otherwise would make the output very difficult to digest due to information overload.

### Load backtest results to pandas dataframe

Analyze a trades dataframe (also used below for plotting)

In [ ]:
from freqtrade.data.btanalysis import load_backtest_data, load_backtest_stats


# if backtest_dir points to a directory, it'll automatically load the last backtest file.
backtest_dir = config["user_data_dir"] / "backtest_results"
# backtest_dir can also point to a specific file
# backtest_dir = (
#   config["user_data_dir"] / "backtest_results/backtest-result-2020-07-01_20-04-22.json"
# )


In [ ]:
# You can get the full backtest statistics by using the following command.
# This contains all information used to generate the backtest result.
stats = load_backtest_stats(backtest_dir)

strategy = "SampleStrategy"
# All statistics are available per strategy, so if `--strategy-list` was used during backtest,
# this will be reflected here as well.
# Example usages:
print(stats["strategy"][strategy]["results_per_pair"])
# Get pairlist used for this backtest
print(stats["strategy"][strategy]["pairlist"])
# Get market change (average change of all pairs from start to end of the backtest period)
print(stats["strategy"][strategy]["market_change"])
# Maximum drawdown ()
print(stats["strategy"][strategy]["max_drawdown_abs"])
# Maximum drawdown start and end
print(stats["strategy"][strategy]["drawdown_start"])
print(stats["strategy"][strategy]["drawdown_end"])


# Get strategy comparison (only relevant if multiple strategies were compared)
print(stats["strategy_comparison"])


In [ ]:
# Load backtested trades as dataframe
trades = load_backtest_data(backtest_dir)

# Show value-counts per pair
trades.groupby("pair")["exit_reason"].value_counts()


## Plotting daily profit / equity line

In [ ]:
# Plotting equity line (starting with 0 on day 1 and adding daily profit for each backtested day)

import pandas as pd
import plotly.express as px

from freqtrade.configuration import Configuration
from freqtrade.data.btanalysis import load_backtest_stats


# strategy = 'SampleStrategy'
# config = Configuration.from_files(["user_data/config.json"])
# backtest_dir = config["user_data_dir"] / "backtest_results"

stats = load_backtest_stats(backtest_dir)
strategy_stats = stats["strategy"][strategy]

df = pd.DataFrame(columns=["dates", "equity"], data=strategy_stats["daily_profit"])
df["equity_daily"] = df["equity"].cumsum()

fig = px.line(df, x="dates", y="equity_daily")
fig.show()


### Load live trading results into a pandas dataframe

In case you did already some trading and want to analyze your performance

In [ ]:
from freqtrade.data.btanalysis import load_trades_from_db


# Fetch trades from database
trades = load_trades_from_db("sqlite:///tradesv3.sqlite")

# Display results
trades.groupby("pair")["exit_reason"].value_counts()


## Analyze the loaded trades for trade parallelism
This can be useful to find the best `max_open_trades` parameter, when used with backtesting in conjunction with a very high `max_open_trades` setting.

`analyze_trade_parallelism()` returns a timeseries dataframe with an "open_trades" column, specifying the number of open trades for each candle.

In [ ]:
from freqtrade.data.btanalysis import analyze_trade_parallelism


# Analyze the above
parallel_trades = analyze_trade_parallelism(trades, "5m")

parallel_trades.plot()


## Plot results

Freqtrade offers interactive plotting capabilities based on plotly.

In [ ]:
from freqtrade.plot.plotting import generate_candlestick_graph


# Limit graph period to keep plotly quick and reactive

# Filter trades to one pair
trades_red = trades.loc[trades["pair"] == pair]

data_red = data["2019-06-01":"2019-06-10"]
# Generate candlestick graph
graph = generate_candlestick_graph(
    pair=pair,
    data=data_red,
    trades=trades_red,
    indicators1=["sma20", "ema50", "ema55"],
    indicators2=["rsi", "macd", "macdsignal", "macdhist"],
)


In [ ]:
# Show graph inline
# graph.show()

# Render graph in a separate window
graph.show(renderer="browser")


## Plot average profit per trade as distribution graph

In [ ]:
import plotly.figure_factory as ff


hist_data = [trades.profit_ratio]
group_labels = ["profit_ratio"]  # name of the dataset

fig = ff.create_distplot(hist_data, group_labels, bin_size=0.01)
fig.show()


Feel free to submit an issue or Pull Request enhancing this document if you would like to share ideas on how to best analyze the data.